© 2025 Mobile Perception Systems Lab at TU/e. All rights reserved. Licensed under the MIT License.

PRE SETUP


In [5]:
from google.colab import drive, userdata

drive.mount('/content/drive')

TOKEN = userdata.get('GITHUB_TOKEN')
REPO = "MaskArchitectureAnomaly_CourseProject"

!git clone https://{TOKEN}@github.com/filoppos/MaskArchitectureAnomaly_CourseProject.git /content/project
#!git pull origin main
#%cd /content/project/eomt


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path '/content/project' already exists and is not an empty directory.


In [21]:
!git config --global user.email "filippo2003.manca@gmail.com"
!git config --global user.name "filoppos"

In [7]:
%cd /content/project/eomt


/content/project/eomt


In [8]:
!pip install -q -r requirements.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.4/219.4 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 35.0 MB/s 

In [9]:
!ls

configs   docs		   __init__.py	main.py  README.md	   training
datasets  inference.ipynb  LICENSE	models	 requirements.txt


## Setup

In [10]:
import yaml
from lightning import seed_everything
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
import warnings
import importlib

seed_everything(0, verbose=False)

TASK = "coco"  # "cityscapes" oppure "coco"

device = 0  # TODO: change to the GPU you want to use
img_idx = 0  # TODO: change to the index of the image you want to visualize
data_path = "/content/drive/MyDrive/Fundamentals Progetto/Coding_Part_Project"  # TODO: change to the dataset directory

config_path = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# Carica config COCO solo per i parametri del modello
coco_config_path = "configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
with open(coco_config_path, "r") as f:

    coco_config = yaml.safe_load(f)


def create_mapping(images, ignore_index):
    unique_ids = np.unique(np.concatenate([np.unique(img) for img in images]))
    valid_ids = unique_ids[unique_ids != ignore_index]
    colors = np.array(
        [plt.cm.hsv(i / len(valid_ids))[:3] for i in range(len(valid_ids))]
    )
    mapping = {cid: colors[i] for i, cid in enumerate(valid_ids)}
    mapping[ignore_index] = np.array([0, 0, 0])
    return mapping


def apply_colormap(image, mapping):
    colored_image = np.zeros((*image.shape, 3))
    for cid in np.unique(image):
        colored_image[image == cid] = mapping.get(cid, [0, 0, 0])
    return colored_image

## Load dataset

Ensure the dataset files are correctly prepared and placed in the folder specified by `data_path`.

In [11]:
import os
os.chdir('/content/project/eomt')
print(os.getcwd())  # deve stampare /content/project/eomt

/content/project/eomt


In [12]:
#%cd /content/drive/MyDrive/Fundamentals Progetto/Coding_Part_Project

In [13]:
!ls

configs   docs		   __init__.py	main.py  README.md	   training
datasets  inference.ipynb  LICENSE	models	 requirements.txt


In [14]:
data_module_name, class_name = config["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = config["data"].get("init_args", {})

data = data_module(
    path=data_path,
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
    **data_module_kwargs
).setup()

In [15]:
import gc
import torch

# Libera memoria GPU
del model
gc.collect()
torch.cuda.empty_cache()
print(f"Memoria GPU libera: {torch.cuda.mem_get_info()[0]/1024**3:.2f} GB")

## Load model

In [16]:
warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

active_config = config if TASK == "cityscapes" else coco_config

# Istanzia un data module temporaneo solo per leggere i parametri del modello COCO
if TASK == "coco":
    coco_data_module_name, coco_class_name = coco_config["data"]["class_path"].rsplit(".", 1)
    coco_data_cls = getattr(importlib.import_module(coco_data_module_name), coco_class_name)
    coco_data_kwargs = coco_config["data"].get("init_args", {})
    model_data = coco_data_cls(path=data_path, batch_size=1, num_workers=0,
                               check_empty_targets=False, **coco_data_kwargs)
else:
    model_data = data  # Cityscapes, già caricato

# Ora usi model_data.num_classes e model_data.img_size come faceva il prof
encoder_cfg = active_config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=model_data.img_size, **encoder_cfg.get("init_args", {}))

network_cfg = active_config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
network = network_cls(
    masked_attn_enabled=False,
    num_classes=model_data.num_classes,
    encoder=encoder,
    **network_kwargs,
)

lit_module_name, lit_class_name = active_config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in active_config["model"]["init_args"].items() if k != "network"}
if "stuff_classes" in active_config["data"].get("init_args", {}):
    model_kwargs["stuff_classes"] = active_config["data"]["init_args"]["stuff_classes"]

model = (
    lit_cls(
        img_size=model_data.img_size,
        num_classes=model_data.num_classes,
        network=network,
        **model_kwargs,
    )
    .eval()
    .to(device)
)
print(f"Modello {TASK}: img_size={model_data.img_size}, num_classes={model_data.num_classes}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

## Load pre-trained weights from Hugging Face Hub (Dal drive invece di huggingFace c'è li hanno dati loro)
The model weights are downloaded from the Hugging Face Hub using the logger name from the config. Make sure you have a working internet connection.

In [ ]:
!ls

In [18]:
WEIGHTS = {
    "cityscapes": "/content/drive/MyDrive/Fundamentals Progetto/Coding_Part_Project/CourseProjectAnomaly/eomt_cityscapes.bin",
    "coco":       "/content/drive/MyDrive/Fundamentals Progetto/Coding_Part_Project/CourseProjectAnomaly/eomt_coco.bin",
}

state_dict = torch.load(WEIGHTS[TASK], map_location=f"cuda:{device}", weights_only=True)
model.load_state_dict(state_dict, strict=False)
print(f"Pesi {TASK} caricati!")

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

##Mapping COCO --> CityScapes

Cityscapes 19 classi (tutte urban driving):
road, sidewalk, building, wall, fence, pole, traffic light, traffic sign, vegetation, terrain, sky, person, rider, car, truck, bus, train, motorcycle, bicycle
Cityscapes train IDs: road=0, sidewalk=1, building=2, wall=3, fence=4, pole=5, traffic light=6, traffic sign=7, vegetation=8, terrain=9,
sky=10, person=11, rider=12, car=13, truck=14, bus=15, train=16, motorcycle=17, bicycle=18

In [ ]:
# Per vedere i nomi delle classi COCO con i loro indici
#import json
# Scarica la lista ufficiale
#!wget -q https://raw.githubusercontent.com/cocodataset/panopticapi/master/panoptic_coco_categories.json -O /tmp/coco_cats.json
#
#with open("/tmp/coco_cats.json") as f:
#    coco_cats = json.load(f)
#
#for i, cat in enumerate(coco_cats):
#    print(f"idx={i:3d}  id={cat['id']:3d}  supercategory={cat['supercategory']:15s}  name={cat['name']}")

In [ ]:
# ── COCO panoptic idx (0-based) → Cityscapes train ID ──
# Cityscapes: road=0, sidewalk=1, building=2, wall=3, fence=4,
# pole=5, traffic light=6, traffic sign=7, vegetation=8, terrain=9,
# sky=10, person=11, rider=12, car=13, truck=14, bus=15, train=16,
# motorcycle=17, bicycle=18

COCO_TO_CITYSCAPES = {
    # THINGS (0-79)
    0:  11,  # person
    1:  18,  # bicycle
    2:  13,  # car
    3:  17,  # motorcycle
    5:  15,  # bus
    6:  16,  # train
    7:  14,  # truck
    9:   6,  # traffic light
    11:  7,  # stop sign → traffic sign

    # STUFF (80-132)
    100:  0,  # road
    123:  1,  # pavement-merged → sidewalk
    129:  2,  # building-other-merged
    91:   2,  # house → building
    82:   2,  # bridge → building (approssimazione)
    101:  2,  # roof → building
    107:  2,  # tent → building
    86:   2,   # door-stuff → building
    118:  2,   # ceiling-merged → building
    115:  2,   # window-other → building
    114:  2,   # window-blind → building
    109:  3,  # wall-brick → wall
    110:  3,  # wall-stone → wall
    111:  3,  # wall-tile → wall
    112:  3,  # wall-wood → wall
    131:  3,  # wall-other-merged → wall
    117:  4,  # fence-merged → fence
    74:  7,   # clock → traffic sign (approssimazione)
    116:  8,  # tree-merged → vegetation
    88:   8,  # flower → vegetation
    90:   9,  # gravel → terrain
    96:   9,  # platform → terrain
    97:   9,  # playingfield → terrain
    98:   9,  # railroad → terrain
    102:  9,  # sand → terrain
    126:  9,  # dirt-merged → terrain
    125:  9,  # grass-merged → terrain (era 8)
    119: 10,  # sky-other-merged → sky
}

VOID = 255  # tutto il resto → ignorato nella valutazione

In [ ]:
def map_coco_to_cityscapes(pred_coco):
    """
    pred_coco: np.array H×W con indici COCO (0-132)
    ritorna: np.array H×W con indici Cityscapes (0-18) o 255 (void)
    """
    out = np.full_like(pred_coco, VOID)
    for coco_idx, city_idx in COCO_TO_CITYSCAPES.items():
        out[pred_coco == coco_idx] = city_idx
    return out

In [ ]:
CITYSCAPES_NAMES = [
    'road', 'sidewalk', 'building', 'wall', 'fence', 'pole',
    'traffic light', 'traffic sign', 'vegetation', 'terrain',
    'sky', 'person', 'rider', 'car', 'truck', 'bus', 'train',
    'motorcycle', 'bicycle'
]  # cosi capiamo i dettagli per classe e dove va male

def compute_miou(pred, target, num_classes=19, ignore_index=255, return_per_class=False):
    ious = []
    per_class = {}
    for cls in range(num_classes):
        pred_mask   = (pred == cls)
        target_mask = (target == cls)
        valid_mask  = (target != ignore_index)

        intersection = (pred_mask & target_mask & valid_mask).sum()
        union        = ((pred_mask | target_mask) & valid_mask).sum()

        if union == 0:
            continue
        iou = intersection / union
        ious.append(iou)
        per_class[CITYSCAPES_NAMES[cls]] = iou

    miou = np.mean(ious) if ious else 0.0
    if return_per_class:
        return miou, per_class
    return miou

## Semantic inference (pixel-wise classification)

> This inference method also works when applied to a model trained for panoptic segmentation.

Semantic inference computes per-pixel class scores by combining mask and class predictions:

$$
\sum_i p_i(c) \cdot m_i[h, w]
$$

Here, $p_i(c)$ is the class probability for class $c$ (excluding "no object"), and $m_i[h, w]$ is the sigmoid-normalized mask value for query $i$ at pixel $(h, w)$. The final class is selected by taking the argmax over classes.  
  
*This inference method was originally introduced in MaskFormer.*

In [ ]:
IGNORE_INDEX = 255


def infer_semantic(img, target):
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]
        crops, origins = model.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], data.img_size, mode="bilinear"
        )

        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        preds = logits[0].argmax(0).cpu()

    pred_array = preds.numpy()
    target_array = model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[
        0
    ].numpy()
    return pred_array, target_array


def plot_semantic_results(img, pred_array, target_array):
    mapping = create_mapping([pred_array, target_array], IGNORE_INDEX)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img.permute(1, 2, 0).cpu().numpy())
    axes[0].set_title("Image")
    axes[1].imshow(apply_colormap(pred_array, mapping))
    axes[1].set_title("Prediction")
    axes[2].imshow(apply_colormap(target_array, mapping))
    axes[2].set_title("Target")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


img, target = data.val_dataloader().dataset[img_idx]
pred_array, target_array = infer_semantic(img, target)
plot_semantic_results(img, pred_array, target_array)

## Panoptic inference (segmentation with instance IDs)

> This inference method also works when applied to a model trained for instance segmentation.

Panoptic inference assigns each pixel $[h, w]$ to the query $i$ that maximizes the product of class and mask confidence:

$$
p_i(c_i) \cdot m_i[h, w]
$$

where $c_i = \arg\max_c \, p_i(c)$ is the most likely class for query $i$. A pixel is assigned to a query only if both the class confidence and mask confidence are high. Pixels assigned to the same query form a segment labeled with $c_i$. "Stuff" segments with the same class are merged; "thing" segments are kept distinct using the query index. Low-confidence and heavily occluded predictions are filtered out.  
  
*This inference method was originally introduced in MaskFormer.*

In [ ]:
def infer_panoptic(img, target):
    torch.cuda.empty_cache()  #
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]

        transformed_imgs = model.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(transformed_imgs)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], model.img_size, mode="bilinear"
        )
        mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(
            mask_logits, img_sizes
        )

        preds = model.to_per_pixel_preds_panoptic(
            mask_logits,
            class_logits_per_layer[-1],
            model.stuff_classes,
            model.mask_thresh,
            model.overlap_thresh,
        )[0].cpu()

    pred = preds.numpy()
    sem_pred, inst_pred = pred[..., 0], pred[..., 1]

    target_seg = model.to_per_pixel_targets_panoptic([target])[0].cpu().numpy()
    sem_target, inst_target = target_seg[..., 0], target_seg[..., 1]

    return sem_pred, inst_pred, sem_target, inst_target


def draw_black_border(sem, inst, mapping):
    h, w = sem.shape
    out = np.zeros((h, w, 3))
    for s in np.unique(sem):
        out[sem == s] = mapping[s]

    combined = sem.astype(np.int64) * 100000 + inst.astype(np.int64)
    border = np.zeros((h, w), dtype=bool)
    border[1:, :] |= combined[1:, :] != combined[:-1, :]
    border[:-1, :] |= combined[1:, :] != combined[:-1, :]
    border[:, 1:] |= combined[:, 1:] != combined[:, :-1]
    border[:, :-1] |= combined[:, 1:] != combined[:, :-1]
    out[border] = 0
    return out


def plot_panoptic_results(img, sem_pred, inst_pred, sem_target, inst_target):
    all_ids = np.union1d(np.unique(sem_pred), np.unique(sem_target))
    mapping = {
        s: (
            [0, 0, 0]
            if s == -1 or s == model.num_classes
            else plt.cm.hsv(i / len(all_ids))[:3]
        )
        for i, s in enumerate(all_ids)
    }

    vis_pred = draw_black_border(sem_pred, inst_pred, mapping)
    vis_target = draw_black_border(sem_target, inst_target, mapping)

    img_np = (
        img.cpu().numpy().transpose(1, 2, 0) if img.dim() == 3 else img.cpu().numpy()
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_np)
    axes[0].set_title("Input")
    axes[1].imshow(vis_pred)
    axes[1].set_title("Prediction")
    axes[2].imshow(vis_target)
    axes[2].set_title("Target")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


img, target = data.val_dataloader().dataset[img_idx]
sem_pred, inst_pred, sem_target, inst_target = infer_panoptic(img, target)
plot_panoptic_results(img, sem_pred, inst_pred, sem_target, inst_target)

#Compute mIoU
After the visualizations

In [ ]:
from tqdm import tqdm

def evaluate_and_print(model, dataloader, is_coco=False):
    all_ious = []
    all_per_class = {name: [] for name in CITYSCAPES_NAMES}

    for batch in tqdm(dataloader):
        img, target = batch
        if isinstance(img, list): img = img[0]
        if isinstance(target, list): target = target[0]

        if is_coco:
            sem_pred, _, _, _ = infer_panoptic(img, target)
            pred_array = map_coco_to_cityscapes(sem_pred)
            target_array = model.to_per_pixel_targets_semantic(
                [target], IGNORE_INDEX)[0].numpy()
        else:
            pred_array, target_array = infer_semantic(img, target)

        miou, per_class = compute_miou(pred_array, target_array, return_per_class=True)
        all_ious.append(miou)
        for name, iou in per_class.items():
            all_per_class[name].append(iou)

    # Stampa risultati
    print(f"\n{'Classe':<20} {'IoU':>6}")
    print("-" * 28)
    results = {name: np.mean(v) for name, v in all_per_class.items() if v}
    for name, iou in sorted(results.items(), key=lambda x: x[1]):
        print(f"{name:<20} {iou:.4f}")
    print("-" * 28)
    print(f"{'mIoU':<20} {np.mean(all_ious):.4f}")

evaluate_and_print(model, data.val_dataloader(), is_coco=(TASK == "coco"))

In [25]:
# Pulisce i metadati widget che rompono il rendering GitHub
import json

notebook_path = "/content/project/eomt/inference.ipynb"  # aggiusta se serve
with open(notebook_path, "r") as f:
    nb = json.load(f)

# Rimuove metadata.widgets che causa il problema
if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

with open(notebook_path, "w") as f:
    json.dump(nb, f, indent=1)

print("✅ Metadata puliti")

✅ Metadata puliti
